# Continuous Speech Recognition (CSR)

In this part of the project, we will build a complete automatic speech recognition (ASR) system for continuous speech using a pretrained self-supervised speech model, [HuBERT](https://arxiv.org/abs/2106.07447).

For training, we will use PyTorch's built-in **CTC loss** function, which is commonly used for sequence-to-sequence alignment in ASR.

For data preparation and batching, we will use [Lhotse](https://lhotse.readthedocs.io/), a toolkit for working with speech datasets and building efficient training pipelines.

### Install required packages

In [ ]:
! pip install torch==1.12.1 torchvision==0.13.1 torchaudio==0.12.1 torchtext==0.13.1
! pip install transformers
! pip install lhotse
! pip install sentencepiece>=0.1.96
! pip install editdistance
! pip install jiwer

# The ASR Model

We are going to build a very simple ASR model. Modern ASR systems use Deep Neural Networks in a number of ways. However, all approaches share one basic component: the **Encoder**.

The encoder is responsible for taking raw speech, or speech features, and encoding them into a vector representation. Ideally this vector representation will have either removed all distractor variables (background noise, channel information), or it will have partitioned the space such that sounds that change the meaning of a sentence (i.e., the transcript), will be separable.

When two different speech sounds, also known as **phones**, result in utterances, or words with different meanings we say that they are phonemically contrastive. These phones are also known as *minimal pairs*. If you can swap one phone for another without changing the meaning of a word, we call those two phones *allophones*. A **phoneme** in a language is the set of all phones that are allophonic. Thus, we see that the encoder in ASR is probably learning a representation of phonemes.
If you feel confused about these terms checkout this very nice explanation and comparison between phone, phoneme, and allophone https://www.theedadvocate.org/the-differences-between-a-phone-phoneme-and-an-allophone/

As in most NLP, and ML disciplines, the transformer architecture is commonly used for the speech encoder. The one difference in speech is that the bit-rate of the signal is much higher, and we need to significantly downsample the speech before encoding it. For this, we either use the Short-Time Fourier Transform (STFT) over windows of speech, or we use learned filters which accomplish a very similar task. These learned filters are convolutional neural networks and you will almost always see a few convolution layers at the input of speech encoders. Commonly the frame-rate is reduced from 16 kHz, the sampling rate of the raw audio signal, to about 100 Hz, or even to 25 Hz at the output of the encoder. In this project we will be using an output rate of 50 Hz, i.e., we will downsample by a factor of 320x when using a 16 kHz audio.

### 1. **First lets download and prepare ASR dataset**

We will mainly use lhotse https://lhotse.readthedocs.io/ to load and prepare the dataset. In lhotse we will need two types of manifests to fully describe the audio corpora: a recording manifest and a supervision manifest.

**Recording** manifest describes the recordings in a given corpus. It contains information about the recording, such as its path(s), duration, the number of samples, etc.

**Supervision** represents a time interval (segment) annotated with some supervision labels and/or metadata, such as the transcription, the speaker identity, the language, etc.

For more details you can check https://lhotse.readthedocs.io/en/latest/corpus.html

In [ ]:
from pathlib import Path

from lhotse import CutSet
from lhotse.recipes import prepare_librispeech, download_librispeech


def get_data(data_dir='/content/data', download=False):
    train_set = "train-clean-5"   # Mini LibriSpeech train split
    dev_set = "dev-clean-2"       # Mini LibriSpeech dev split
    test_set = "test-clean"       # Full LibriSpeech test split

    data_dir = Path(data_dir)

    if download:
        corpus_dir = download_librispeech(
            target_dir=data_dir,
            dataset_parts=[train_set, dev_set, test_set],
        )
    else:
        corpus_dir = data_dir if data_dir.name == "LibriSpeech" else data_dir / "LibriSpeech"

    manifests = prepare_librispeech(
        corpus_dir=corpus_dir,
        dataset_parts=[train_set, dev_set, test_set],
    )

    cuts_train = CutSet.from_manifests(**manifests[train_set])
    cuts_dev = CutSet.from_manifests(**manifests[dev_set])
    cuts_test = CutSet.from_manifests(**manifests[test_set])

    cuts_train = cuts_train.trim_to_supervisions(keep_overlapping=False)
    cuts_dev = cuts_dev.trim_to_supervisions(keep_overlapping=False)
    cuts_test = cuts_test.trim_to_supervisions(keep_overlapping=False)

    return cuts_train, cuts_dev, cuts_test

In [ ]:
# Download the data and check the cutsets
cuts_train, cuts_dev, cuts_test = get_data(download=True)

In [ ]:
# Explore the loaded cutsets
cuts_train = cuts_train.to_eager()
cuts = list(cuts_train)
print(f"Number of segments in train set: {len(cuts)}\n")

print(f"first segments id: {cuts[0].id}")
print(f"first segment supervision:\n {cuts[0].supervisions}\n")
print(f"first segment recording:\n {cuts[0].recording}\n")

print(f"describe the train segments statistics: ")
cuts_train.describe()

# Exploring Lhotse
Lhotse is very flexible and allows us to do many kinds of audio processing. Here we show just a few examples of the kinds of things you can do.

Feel free to continue exploring some other lhotse functionality!
https://github.com/lhotse-speech/lhotse/blob/master/examples/01-cut-python-api.ipynb

In [ ]:
# Plot audio
cuts_train[10].plot_audio()

In [ ]:
# Compute Features
from lhotse import CutSet, Mfcc, Fbank
cuts_subset = cuts_train.subset(first=2).compute_and_store_features(
    Fbank(), "/content/data", num_jobs=1
)
cut = cuts_subset[0]
print(cut.supervisions[0].text)
cut.plot_features()

### 2. **Create pytorch dataset class**

In [ ]:
from lhotse.dataset.input_strategies import AudioSamples
from lhotse.workarounds import Hdf5MemoryIssueFix
from lhotse.dataset.speech_recognition import validate_for_asr
from lhotse.dataset.collation import TokenCollater
import torch

class ASRDataset(torch.utils.data.Dataset):
    def __init__(self, tokenizer: TokenCollater):
        self.tokenizer = tokenizer
        self.input_method = AudioSamples()
        self.hdf5_fix = Hdf5MemoryIssueFix(reset_interval=100)

    def __getitem__(self, cuts: CutSet) -> dict:
        validate_for_asr(cuts)
        self.hdf5_fix.update()
        cuts = cuts.sort_by_duration(ascending=False)
        inputs, _ = self.input_method(cuts)
        supervision_intervals = self.input_method.supervision_intervals(cuts)
        tokens, token_lens = self.tokenizer(cuts)
        return {
            'input': inputs,
            'target': tokens,
            'target_lens': token_lens,
            'lens': supervision_intervals,
            'text': [s.text for cut in cuts for s in cut.supervisions]
        }

    def move_to(b, device):
        return {
            'input': b['input'].to(device),
            'target': b['target'].to(device),
            'target_lens': b['target_lens'].to(device),
            'lens': {
                'sequence_idx': b['lens']['sequence_idx'].to(device),
                'start_sample': b['lens']['start_sample'].to(device),
                'num_samples': b['lens']['num_samples'].to(device)
            },
            'text': b['text'],
        }

In [ ]:
from typing import List, Tuple

import numpy as np
import torch
from lhotse import CutSet


class CharacterTokenizer:
    """
    Collate character-level transcripts from a CutSet into a padded tensor.

    This class:
      1) builds a character vocabulary from the provided cuts,
      2) converts each transcript into a sequence of character IDs,
      3) pads all sequences in a batch to the same length,
      4) returns the padded tensor and the true sequence lengths.

    Notes:
      - This is character-level tokenization, not word-level or BPE.
      - For CTC training, it is usually better to set add_bos=False and add_eos=False.
      - The vocabulary includes special symbols like <pad> and <unk>.
    """

    def __init__(
        self,
        cuts: CutSet,
        add_eos: bool = True,
        add_bos: bool = True,
        pad_symbol: str = "<pad>",
        unk_symbol: str = "<unk>",
    ):
        self.pad_symbol = pad_symbol
        self.unk_symbol = unk_symbol
        self.add_bos = add_bos
        self.add_eos = add_eos

        # Collect all unique characters that appear in the supervision text.
        tokens = set()
        for cut in cuts:
            text = self._cut_to_text(cut)
            tokens.update(text)

        # Build the full vocabulary by combining special symbols and characters.
        # CTC blank symbol is placed at index 0.
        specials = ["<blk>", pad_symbol, unk_symbol]
        if add_bos:
            specials.append("<bos>")
        if add_eos:
            specials.append("<eos>")

        self.idx2token = specials + sorted(tokens)  # sorted for determinism

        # Create mappings between tokens and integer IDs.
        self.token2idx = {tok: i for i, tok in enumerate(self.idx2token)}

        # Store the IDs of the special symbols for later use.
        self.pad_id = self.token2idx[pad_symbol]
        self.unk_id = self.token2idx[unk_symbol]
        self.blk_id = self.token2idx["<blk>"]
        self.bos_id = self.token2idx.get("<bos>", -1)
        self.eos_id = self.token2idx.get("<eos>", -1)

    def __len__(self) -> int:
        return len(self.idx2token)

    def _cut_to_text(self, cut) -> str:
        """Convert a cut to a single transcript string."""
        return " ".join(s.text for s in cut.supervisions).strip()

    def encode_text(self, text: str) -> List[int]:
        """Convert a text string into a list of character IDs."""
        ids = []
        if self.add_bos and self.bos_id >= 0:
            ids.append(self.bos_id)
        for ch in text:
            ids.append(self.token2idx.get(ch, self.unk_id))
        if self.add_eos and self.eos_id >= 0:
            ids.append(self.eos_id)
        return ids

    def __call__(self, cuts: CutSet) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Convert a batch of cuts into padded token sequences.

        Returns:
            tokens_batch (torch.LongTensor): Shape (B, L), padded token IDs.
            tokens_lens (torch.LongTensor): Shape (B,), true sequence lengths.
        """
        sequences = [self.encode_text(self._cut_to_text(cut)) for cut in cuts]
        tokens_lens = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
        max_len = tokens_lens.max().item()
        padded = torch.full((len(sequences), max_len), self.pad_id, dtype=torch.long)
        for i, seq in enumerate(sequences):
            padded[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        return padded, tokens_lens

    def inverse(
        self,
        tokens_batch: list,
        tokens_lens: list,
    ) -> List[str]:
        """
        Reconstruct strings from a padded batch of token IDs.

        Args:
            tokens_batch: torch.LongTensor of shape (B, L).
            tokens_lens:  torch.LongTensor of shape (B,).

        Returns:
            sentences: List of length B with the reconstructed strings.
        """
        sentences = []
        for ids, length in zip(tokens_batch, tokens_lens):
            if isinstance(ids, torch.Tensor):
                ids = ids.tolist()
            if isinstance(length, torch.Tensor):
                length = length.item()
            valid_ids = ids[:length]
            chars = []
            for tid in valid_ids:
                if tid in (self.pad_id, self.blk_id, self.bos_id, self.eos_id):
                    continue
                tok = self.idx2token[tid]
                if tok not in ("<pad>", "<blk>", "<bos>", "<eos>", "<unk>"):
                    chars.append(tok)
            sentences.append("".join(chars))
        return sentences

# The HuBERT Encoder

The encoder we are using is a pretrained model called HuBERT which is trained on a large amount of unlabeled data, and the learned representations have been shown to encode phonemic information extremely well.

In [ ]:
from transformers import HubertModel
import torch.nn as nn


class SpeechEncoder(nn.Module):
    def __init__(self, num_classes, freeze_updates=1000):
        super(SpeechEncoder, self).__init__()

        self.encoder = HubertModel.from_pretrained('facebook/hubert-base-ls960')
        self.encoder.feature_extractor._freeze_parameters()
        self.linear = nn.Linear(self._get_output_dim(), num_classes)
        self.freeze_updates = freeze_updates

    def _get_output_dim(self):
        '''Pass a dummy tensor through the encoder to determine its output size.
        Min input length is 400 samples (320x downsampling of HuBERT CNN stack).'''
        x = torch.rand(1, 400)
        return self.encoder(x).last_hidden_state.size(-1)

    def forward(self, mb, iter_num=0):
        """
        Run the model on one minibatch.

        Args:
            mb: dict with keys:
                'input' (FloatTensor): shape (B, T) — raw waveform
                'target' (LongTensor): shape (B, L)
                'lens'['num_samples'] (IntTensor): shape (B,)
            iter_num: current training step (used to decide encoder freezing).

        Returns:
            y (FloatTensor): shape (B, T_out, C) — output logits.
            input_lens (IntTensor): shape (B,) — frame counts after CNN downsampling.
        """
        ### TODO: Implement the forward pass of SpeechEncoder.
        # Steps:
        #   1. Choose a context manager based on iter_num vs. self.freeze_updates:
        #        scope = torch.no_grad() if iter_num < self.freeze_updates
        #               else torch.enable_grad()
        #      (This freezes the encoder parameters during early warmup steps.)
        #   2. Inside `with scope:`:
        #        a. x = mb['input']  # (B, T)
        #        b. If x.dim() == 3 (shape B, T, 1), squeeze the last dimension.
        #        c. encoder_out = self.encoder(x).last_hidden_state  # (B, T_enc, D)
        #   3. Apply self.linear to project encoder output to logits:
        #        y = self.linear(encoder_out)  # (B, T_enc, num_classes)
        #
        # The CNN downsampling formula below is provided — do NOT modify it.
        input_lens = mb['lens']['num_samples']
        for width, stride in [(10, 5), (3, 2), (3, 2), (3, 2), (3, 2), (2, 2), (2, 2)]:
            input_lens = torch.floor((input_lens - width) / stride + 1).int()
        # Replace this line with:  return y, input_lens
        raise NotImplementedError("Implement SpeechEncoder.forward")

In [ ]:
#@title show solution { display-mode: "form" }
from transformers import HubertModel
import torch.nn as nn


class SpeechEncoder(nn.Module):
    def __init__(self, num_classes, freeze_updates=1000):
        super(SpeechEncoder, self).__init__()
        self.encoder = HubertModel.from_pretrained('facebook/hubert-base-ls960')
        self.encoder.feature_extractor._freeze_parameters()
        self.linear = nn.Linear(self._get_output_dim(), num_classes)
        self.freeze_updates = freeze_updates

    def _get_output_dim(self):
        x = torch.rand(1, 400)
        return self.encoder(x).last_hidden_state.size(-1)

    def forward(self, mb, iter_num=0):
        scope = torch.no_grad() if iter_num < self.freeze_updates else torch.enable_grad()
        with scope:
            x = mb['input']
            if x.dim() == 3:
                x = x.squeeze(-1)
            encoder_out = self.encoder(x).last_hidden_state

        y = self.linear(encoder_out)

        input_lens = mb['lens']['num_samples']
        for width, stride in [(10, 5), (3, 2), (3, 2), (3, 2), (3, 2), (2, 2), (2, 2)]:
            input_lens = torch.floor((input_lens - width) / stride + 1).int()
        return y, input_lens

# CTC Loss
This is a wrapper around PyTorch's CTC loss function.
CTC (Connectionist Temporal Classification) allows training a sequence model without needing frame-level alignments between audio and transcript.

In [ ]:
import torch.nn.functional as F


class CTCLoss(nn.Module):
    def __init__(self):
        super(CTCLoss, self).__init__()

    def forward(self, outputs, targets):
        """
        Compute the CTC loss.

        Args:
            outputs: (logits, output_lengths)
                logits: FloatTensor of shape (B, T, C)
                output_lengths: IntTensor of shape (B,)
            targets: (target_ids, target_lengths)
                target_ids: LongTensor of shape (B, L)
                target_lengths: LongTensor of shape (B,)

        Returns:
            loss (Tensor): scalar CTC loss.
        """
        ### TODO: Implement CTCLoss.forward.
        # Steps:
        #   1. Unpack outputs → (logits, output_lengths)
        #      Unpack targets → (targets_ids, target_lengths)
        #   2. log_probs = logits.log_softmax(dim=-1)   # (B, T, C)
        #   3. Permute log_probs to (T, B, C) — required by F.ctc_loss.
        #   4. loss = F.ctc_loss(log_probs, targets_ids, output_lengths, target_lengths,
        #                        blank=0, reduction='mean', zero_infinity=True)
        #   5. Return loss.
        # Docs: https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.ctc_loss.html
        raise NotImplementedError("Implement CTCLoss.forward")

In [ ]:
#@title show solution { display-mode: "form" }
import torch.nn.functional as F


class CTCLoss(nn.Module):
    def __init__(self):
        super(CTCLoss, self).__init__()

    def forward(self, outputs, targets):
        logits, output_lengths = outputs
        targets_ids, target_lengths = targets
        log_probs = logits.log_softmax(dim=-1)
        log_probs = log_probs.permute(1, 0, 2)
        loss = F.ctc_loss(
            log_probs,
            targets_ids,
            output_lengths,
            target_lengths,
            blank=0,
            reduction='mean',
            zero_infinity=True,
        )
        return loss

In [ ]:
tokenizer = CharacterTokenizer(cuts_train)

In [ ]:
tokenizer.token2idx

# Training Setup

In [ ]:
from lhotse.dataset import DynamicBucketingSampler

# Hyperparameters
num_epochs = 15
min_lr = 1e-07
max_lr = 5e-05
warmup = 4000
freeze = 1000
lr_slope = (max_lr - min_lr) / warmup

# Tokenizer
tokenizer = CharacterTokenizer(cuts_train)

# Datasets & samplers
train_dataset = ASRDataset(tokenizer)
dev_dataset   = ASRDataset(tokenizer)

train_sampler = DynamicBucketingSampler(
    cuts_train, max_duration=80.0, shuffle=True, num_buckets=30,
)
dev_sampler = DynamicBucketingSampler(
    cuts_dev, max_duration=200.0, shuffle=False,
)

train_dloader = torch.utils.data.DataLoader(
    train_dataset, sampler=train_sampler, batch_size=None, num_workers=2,
)
dev_dloader = torch.utils.data.DataLoader(
    dev_dataset, sampler=dev_sampler, batch_size=None, num_workers=2,
)

# Model & loss
model   = SpeechEncoder(len(tokenizer.idx2token), freeze_updates=freeze)
loss_fn = CTCLoss()

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
loss_fn.to(device)

# Optimizer
optim = torch.optim.Adam(
    list(filter(lambda p: p.requires_grad, model.parameters())),
    lr=min_lr,
    weight_decay=1e-06,
)

### Evaluation Metric: WER and CER

During evaluation we report **Word Error Rate (WER)** and **Character Error Rate (CER)**.
Both metrics count substitutions, deletions, and insertions relative to the reference transcript.
We use the [jiwer](https://jitsi.github.io/jiwer/usage/) library.

This function is provided — read through it so you understand what it computes.

In [ ]:
import jiwer
from jiwer.transforms import (
    Compose,
    ToLowerCase,
    RemovePunctuation,
    RemoveMultipleSpaces,
    Strip,
    ReduceToListOfListOfWords,
    ReduceToListOfListOfChars,
)


def compute_wer_cer(references, hypotheses, print_results=True):
    """
    Compute WER and CER with standard normalization.

    Args:
        references  (list[str]): Ground-truth transcriptions.
        hypotheses  (list[str]): Predicted transcriptions.
        print_results (bool): Whether to print formatted results.

    Returns:
        wer_out: jiwer output object (contains substitutions, deletions, insertions, hits).
    """
    word_tr = Compose([
        ToLowerCase(), RemovePunctuation(), RemoveMultipleSpaces(),
        Strip(), ReduceToListOfListOfWords(),
    ])
    char_tr = Compose([
        ToLowerCase(), RemovePunctuation(), RemoveMultipleSpaces(),
        Strip(), ReduceToListOfListOfChars(),
    ])

    wer_out = jiwer.process_words(
        references, hypotheses,
        reference_transform=word_tr, hypothesis_transform=word_tr,
    )
    cer_out = jiwer.process_characters(
        references, hypotheses,
        reference_transform=char_tr, hypothesis_transform=char_tr,
    )

    if print_results:
        print(f"WER: {100 * wer_out.wer:.2f}%  "
              f"(Subs {wer_out.substitutions} / Dels {wer_out.deletions} / Ins {wer_out.insertions})")
        print(f"CER: {100 * cer_out.cer:.2f}%  "
              f"(Subs {cer_out.substitutions} / Dels {cer_out.deletions} / Ins {cer_out.insertions})")

    return wer_out

### CTC Greedy Decoding

Implement the following two functions. They will be called inside the training loop
at the end of each epoch to measure WER/CER on the dev set.

In [ ]:
def ctc_greedy_decode(logits, tokenizer, blank_id=0):
    """
    Greedy CTC decoding.

    Args:
        logits (Tensor): shape (B, T, C) — raw model output.
        tokenizer: has an inverse(tokens_batch, tokens_lens) method.
        blank_id (int): ID of the CTC blank symbol.

    Returns:
        texts (List[str]): one decoded string per item in the batch.
    """
    ### TODO: Implement ctc_greedy_decode.
    # Steps:
    #   1. best_ids = logits.argmax(dim=-1)   # (B, T) — most likely token at each frame
    #   2. For each sequence in best_ids:
    #      a. Iterate through the token IDs.
    #      b. Skip blank tokens (idx == blank_id).
    #      c. Collapse consecutive repeated tokens (skip if idx == prev).
    #      d. Collect the surviving IDs into decoded_ids.
    #   3. Convert decoded_ids to a string with:
    #        tokenizer.inverse([torch.tensor(decoded_ids)], [len(decoded_ids)])[0]
    #      (Use "" for an empty decoded_ids list.)
    #   4. Return the list of decoded strings.
    raise NotImplementedError("Implement ctc_greedy_decode")


def decode_data(model, data_loader, tokenizer):
    """
    Decode a full dataset with CTC greedy decoding.

    Args:
        model: ASR model.
        data_loader: DataLoader over the dataset.
        tokenizer: has an inverse() method.

    Returns:
        hypotheses (List[str]): model predictions.
        references  (List[str]): ground-truth transcripts from batch['text'].
    """
    ### TODO: Implement decode_data.
    # Steps:
    #   1. Set model.eval() and use torch.no_grad().
    #   2. Loop over data_loader batches.
    #   3. Move each batch to device:  batch = ASRDataset.move_to(batch, device)
    #   4. Forward pass: logits, output_lens = model(batch)
    #   5. Decode:  hyps = ctc_greedy_decode(logits, tokenizer)
    #   6. Extend hypotheses with hyps and references with batch['text'].
    #   7. Return (hypotheses, references).
    raise NotImplementedError("Implement decode_data")

In [ ]:
#@title show solution { display-mode: "form" }
def ctc_greedy_decode(logits, tokenizer, blank_id=0):
    best_ids = logits.argmax(dim=-1)  # (B, T)
    texts = []
    for seq_ids in best_ids:
        prev = blank_id
        decoded_ids = []
        for idx in seq_ids:
            idx = idx.item()
            if idx != blank_id and idx != prev:
                decoded_ids.append(idx)
            prev = idx
        if decoded_ids:
            text = tokenizer.inverse(
                [torch.tensor(decoded_ids, dtype=torch.long)],
                [len(decoded_ids)]
            )[0]
        else:
            text = ""
        texts.append(text)
    return texts


def decode_data(model, data_loader, tokenizer):
    model.eval()
    hypotheses, references = [], []
    with torch.no_grad():
        for batch in data_loader:
            batch = ASRDataset.move_to(batch, device)
            logits, output_lens = model(batch)
            hyps = ctc_greedy_decode(logits, tokenizer)
            hypotheses.extend(hyps)
            references.extend(batch['text'])
    return hypotheses, references

### Training loop

In [ ]:
scaler = torch.cuda.amp.GradScaler()
iter_num = 0
curr_lr  = min_lr

### TODO: Implement the main training loop.
#
# For each epoch in range(num_epochs):
#
# ---- TRAINING ----
# 1. model.train()
# 2. Loop over train_dloader:
#    a. Update curr_lr with the warmup / decay schedule:
#         if iter_num < warmup:
#             curr_lr = min_lr + lr_slope * iter_num
#         else:
#             curr_lr = max(min_lr, max_lr * (warmup / iter_num) ** 0.5)
#       Apply curr_lr to all optimizer param groups.
#    b. batch = ASRDataset.move_to(batch, device)
#    c. optim.zero_grad()
#    d. Inside torch.cuda.amp.autocast():
#         outputs = model(batch, iter_num)
#         loss = loss_fn(outputs, (batch['target'], batch['target_lens']))
#    e. scaler.scale(loss).backward()
#       scaler.step(optim)
#       scaler.update()
#    f. Accumulate loss; increment iter_num.
# 3. Print: f"Epoch {e+1}/{num_epochs} - Train Loss: {avg:.4f}, LR: {curr_lr:.6f}"
#
# ---- VALIDATION ----
# 4. model.eval(); torch.no_grad()
# 5. Loop over dev_dloader; accumulate validation loss.
# 6. Print: f"Epoch {e+1}/{num_epochs} - Val Loss: {avg:.4f}"
#
# ---- DECODING & EVALUATION ----
# 7. hypotheses, references = decode_data(model, dev_dloader, tokenizer)
# 8. wer_out = compute_wer_cer(references, hypotheses)

raise NotImplementedError("Implement the training loop")

In [ ]:
#@title show solution { display-mode: "form" }
scaler = torch.cuda.amp.GradScaler()
iter_num = 0
curr_lr  = min_lr

for e in range(num_epochs):
    # ---- TRAINING ----
    model.train()
    total_train_loss, num_train_batches = 0.0, 0

    for batch in train_dloader:
        if iter_num < warmup:
            curr_lr = min_lr + lr_slope * iter_num
        else:
            curr_lr = max(min_lr, max_lr * (warmup / iter_num) ** 0.5)
        for pg in optim.param_groups:
            pg['lr'] = curr_lr

        batch = ASRDataset.move_to(batch, device)
        optim.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(batch, iter_num)
            loss = loss_fn(outputs, (batch['target'], batch['target_lens']))

        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()

        total_train_loss += loss.item()
        num_train_batches += 1
        iter_num += 1

    avg_train = total_train_loss / max(num_train_batches, 1)
    print(f"Epoch {e+1}/{num_epochs} - Train Loss: {avg_train:.4f}, LR: {curr_lr:.6f}")

    # ---- VALIDATION ----
    model.eval()
    total_val_loss, num_val_batches = 0.0, 0
    with torch.no_grad():
        for batch in dev_dloader:
            batch = ASRDataset.move_to(batch, device)
            outputs = model(batch, iter_num)
            loss = loss_fn(outputs, (batch['target'], batch['target_lens']))
            total_val_loss += loss.item()
            num_val_batches += 1

    avg_val = total_val_loss / max(num_val_batches, 1)
    print(f"Epoch {e+1}/{num_epochs} - Val Loss: {avg_val:.4f}")

    # ---- DECODING & EVALUATION ----
    hypotheses, references = decode_data(model, dev_dloader, tokenizer)
    wer_out = compute_wer_cer(references, hypotheses)

In [ ]:
hypotheses, references = decode_data(model, dev_dloader, tokenizer)

In [ ]:
hypotheses[0]

In [ ]:
references[0]

In [ ]:
wer_out = compute_wer_cer(references, hypotheses)

### Analyze the results

In [ ]:
print(jiwer.visualize_alignment(wer_out))

In [ ]:
print(jiwer.visualize_error_counts(wer_out, top_k=10))
subs, ins, dels = jiwer.collect_error_counts(wer_out)

### ASR with BPE

SentencePiece is an unsupervised text tokenizer and detokenizer developed by Google.
First, train a SentencePiece BPE tokenizer on the training transcription text.
You can find instructions for using SentencePiece as a Python module in its documentation here:
https://github.com/google/sentencepiece/blob/master/python/README.md

In [ ]:
!pip install sentencepiece

In [ ]:
import sentencepiece as spm


def write_transcripts(cuts, output_text_path):
    """Write one transcript per line from a CutSet. This text file trains SentencePiece."""
    with open(output_text_path, "w", encoding="utf-8") as f:
        for cut in cuts:
            text = " ".join(s.text for s in cut.supervisions).strip()
            if text:
                f.write(text + "\n")


write_transcripts(cuts_train, "libri5_transcripts.txt")

vocab_size = 100
user_defined_symbols = ["<blk>", "<sos/eos>"]
unk_id     = len(user_defined_symbols)
model_type = "bpe"   # alternatively: "unigram"

spm.SentencePieceTrainer.train(
    input="libri5_transcripts.txt",
    model_prefix="libri5_bpe",
    vocab_size=vocab_size,
    model_type=model_type,
    character_coverage=1.0,
    user_defined_symbols=user_defined_symbols,
    unk_id=unk_id,
    bos_id=-1,
    eos_id=-1,
)

sp = spm.SentencePieceProcessor()
sp.load("libri5_bpe.model")

# Verify encoding / decoding
text = "THIS IS A TEST"
print("Pieces:", sp.encode_as_pieces(text))
print("IDs:",    sp.encode_as_ids(text))
ids = sp.encode_as_ids(text)
print("Decoded:", sp.decode_ids(ids))

Implement a BPE tokenizer similar to the `CharacterTokenizer` above.
It wraps a pretrained SentencePiece model and provides the same interface
(`__call__`, `inverse`, `encode_text`) so it can be used as a drop-in replacement.

The boilerplate (`__init__`, `_build_idx2token`, `__call__`) is provided.
Fill in the two marked methods: **`encode_text`** and **`inverse`**.

In [ ]:
from typing import List, Tuple

import torch
from lhotse import CutSet


class BPETokenizer:
    """
    Collate BPE-tokenized transcripts from a CutSet into a padded tensor.

    Uses a pretrained SentencePiece model. The CTC blank token is assumed to
    be at index 0 (the first user-defined symbol).

    SentencePiece API used here:
      sp.encode(text, out_type=int) -> List[int]
      sp.id_to_piece(i)             -> str
      sp.get_piece_size()           -> int
      sp.pad_id()                   -> int   (may return -1 if not set)
      sp.unk_id()                   -> int
      sp.decode(ids)                -> str
    """

    def __init__(self, bpe_model, unk_id: int = None):
        self.bpe_model = bpe_model
        self.unk_id    = unk_id if unk_id is not None else bpe_model.unk_id()
        self.blk_id    = 0  # CTC blank is the first user-defined symbol
        pad_id = bpe_model.pad_id()
        self.pad_id    = pad_id if pad_id >= 0 else 0
        self.idx2token = self._build_idx2token()
        self.token2idx = {tok: i for i, tok in enumerate(self.idx2token)}

    def _build_idx2token(self) -> List[str]:
        return [self.bpe_model.id_to_piece(i) for i in range(self.bpe_model.get_piece_size())]

    def __len__(self) -> int:
        return self.bpe_model.get_piece_size()

    def _cut_to_text(self, cut) -> str:
        return " ".join(s.text for s in cut.supervisions).strip()

    def encode_text(self, text: str) -> List[int]:
        """Convert a text string into a list of BPE token IDs."""
        ### TODO: Use self.bpe_model.encode(text, out_type=int) and return the result.
        raise NotImplementedError("Implement encode_text")

    def __call__(self, cuts: CutSet) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Convert a batch of cuts into:
          - tokens_batch: LongTensor of shape (B, L) with padded BPE IDs
          - tokens_lens:  LongTensor of shape (B,) with true sequence lengths
        """
        sequences   = [self.encode_text(self._cut_to_text(cut)) for cut in cuts]
        tokens_lens = torch.tensor([len(s) for s in sequences], dtype=torch.long)
        max_len     = tokens_lens.max().item()
        padded      = torch.full((len(sequences), max_len), self.pad_id, dtype=torch.long)
        for i, seq in enumerate(sequences):
            padded[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        return padded, tokens_lens

    def inverse(self, tokens_batch: list, tokens_lens: list) -> List[str]:
        """Reconstruct text strings from a batch of BPE IDs, skipping blank tokens."""
        ### TODO: Implement inverse.
        # For each (ids, length) pair:
        #   1. Convert to Python list if needed; take only ids[:length].
        #   2. Filter out blank tokens: clean_ids = [i for i in ids[:length] if i != self.blk_id]
        #   3. Decode with self.bpe_model.decode(clean_ids) and append to sentences.
        # Return the list of decoded strings.
        raise NotImplementedError("Implement BPETokenizer.inverse")

In [ ]:
# Test the BPE tokenizer
tokenizer_bpe = BPETokenizer(sp)

text = "HELLO WORLD"
ids  = tokenizer_bpe.encode_text(text)
print("Encoded IDs:", ids)
print("Decoded text:", tokenizer_bpe.inverse([ids], [len(ids)]))

Now run the same training pipeline with the BPE tokenizer.
The setup and training loop mirror the character-level version exactly —
compare both runs to see how vocabulary choice affects WER/CER.

In [ ]:
from lhotse.dataset import DynamicBucketingSampler

tokenizer = BPETokenizer(sp)

train_dataset_bpe = ASRDataset(tokenizer)
dev_dataset_bpe   = ASRDataset(tokenizer)

train_sampler_bpe = DynamicBucketingSampler(
    cuts_train, max_duration=80.0, shuffle=True, num_buckets=30,
)
dev_sampler_bpe = DynamicBucketingSampler(
    cuts_dev, max_duration=200.0, shuffle=False,
)

train_dloader_bpe = torch.utils.data.DataLoader(
    train_dataset_bpe, sampler=train_sampler_bpe, batch_size=None, num_workers=2,
)
dev_dloader_bpe = torch.utils.data.DataLoader(
    dev_dataset_bpe, sampler=dev_sampler_bpe, batch_size=None, num_workers=2,
)

asr_model_bpe = SpeechEncoder(len(tokenizer), freeze_updates=freeze)
asr_model_bpe.to(device)

optim_bpe = torch.optim.Adam(
    list(filter(lambda p: p.requires_grad, asr_model_bpe.parameters())),
    lr=min_lr, weight_decay=1e-06,
)

In [ ]:
scaler_bpe   = torch.cuda.amp.GradScaler()
iter_num_bpe = 0
curr_lr_bpe  = min_lr

for e in range(num_epochs):
    # ---- TRAINING ----
    asr_model_bpe.train()
    total_train_loss, num_train_batches = 0.0, 0

    for batch in train_dloader_bpe:
        if iter_num_bpe < warmup:
            curr_lr_bpe = min_lr + lr_slope * iter_num_bpe
        else:
            curr_lr_bpe = max(min_lr, max_lr * (warmup / iter_num_bpe) ** 0.5)
        for pg in optim_bpe.param_groups:
            pg['lr'] = curr_lr_bpe

        batch = ASRDataset.move_to(batch, device)
        optim_bpe.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = asr_model_bpe(batch, iter_num_bpe)
            loss = loss_fn(outputs, (batch['target'], batch['target_lens']))

        scaler_bpe.scale(loss).backward()
        scaler_bpe.step(optim_bpe)
        scaler_bpe.update()

        total_train_loss += loss.item()
        num_train_batches += 1
        iter_num_bpe += 1

    avg_train = total_train_loss / max(num_train_batches, 1)
    print(f"Epoch {e+1}/{num_epochs} - Train Loss (BPE): {avg_train:.4f}, LR: {curr_lr_bpe:.6f}")

    # ---- VALIDATION ----
    asr_model_bpe.eval()
    total_val_loss, num_val_batches = 0.0, 0
    with torch.no_grad():
        for batch in dev_dloader_bpe:
            batch = ASRDataset.move_to(batch, device)
            outputs = asr_model_bpe(batch, iter_num_bpe)
            loss = loss_fn(outputs, (batch['target'], batch['target_lens']))
            total_val_loss += loss.item()
            num_val_batches += 1

    avg_val = total_val_loss / max(num_val_batches, 1)
    print(f"Epoch {e+1}/{num_epochs} - Val Loss (BPE): {avg_val:.4f}")

    # ---- DECODING & EVALUATION ----
    hypotheses, references = decode_data(asr_model_bpe, dev_dloader_bpe, tokenizer)
    wer_out = compute_wer_cer(references, hypotheses)

In [ ]:
hypotheses2, references2 = decode_data(asr_model_bpe, dev_dloader_bpe, tokenizer)
wer_out2 = compute_wer_cer(references2, hypotheses2)

Analyze the results

In [ ]:
print(jiwer.visualize_error_counts(wer_out2, top_k=10))
subs, ins, dels = jiwer.collect_error_counts(wer_out2)